# The Central Limit Theorem — Discrete Case

Take a distribution — *any* distribution, as odd as you like. Draw a fixed number of samples, average them, repeat many times, and plot the averages: you get a **normal distribution**, no matter what you started with. This is the **central limit theorem**, one of the pinnacles of statistics. This notebook builds the discrete case from coin flips: the number of heads in $n$ tosses is a *sum of $n$ Bernoulli variables*, and as $n$ grows its distribution morphs into a Gaussian with mean $np$ and variance $np(1-p)$. It is based on the sixth video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

The normal distribution keeps appearing in unexpected places — and here is the deepest reason of all. Sums (and averages) of many independent random contributions become bell-shaped *regardless of the ingredients' shape*. The law of large numbers said sample means **converge to** $\mu$; the CLT says far more — it describes the entire *shape* of their fluctuations.

We start where the course always starts: with coins.

After completing this notebook you will be able to:

- Recognize the head-count in $n$ flips as a **sum of $n$ i.i.d. Bernoulli variables**.
- Watch the binomial PMF morph from two bars into a bell as $n = 1, 2, 3, 4, 10, \dots$
- Compute the limiting Gaussian's parameters: $\mu = np$, $\sigma^2 = np(1-p)$.
- Use the normal approximation to the binomial — and know when it is trustworthy.

## 2. Theory

### The setup: counting heads

Flip a fair coin once: $X = 1$ for heads, $0$ for tails, each with probability $\tfrac12$ — two bars of height $0.5$. Nothing bell-shaped about it.

Now let $X$ = the number of heads in $n$ flips, and watch the distribution as $n$ increases: with 2 coins, three bars; with 3 coins, four; with 4, five; and by $n = 10$ the PMF *looks a lot like a Gaussian*.

### Why: sums of Bernoullis

Counting heads in $n$ flips is the same as **adding $n$ Bernoulli variables** — each equal to 1 for heads, 0 for tails. And this is the central limit theorem in action:

> As you increase the number of variables you are adding, the distribution of the sum looks more and more like the Gaussian distribution.

The starting distribution (two spikes!) could hardly be less normal — the bell emerges purely from the *adding*.

### The parameters of the limit

For $n$ coins with heads-probability $p$ (the binomial distribution of Lesson 2):

$$
\mu = np \qquad\qquad \sigma^2 = np(1-p)
$$

The video's table for the fair coin ($p = 0.5$):

| $n$ | 1 | 2 | 3 | 4 | 10 |
|---|---|---|---|---|---|
| $\mu = np$ | 0.5 | 1 | 1.5 | 2 | 5 |
| $\sigma^2 = np(1-p)$ | 0.25 | 0.5 | 0.75 | 1 | 2.5 |

And the punchline: as $n$ becomes sufficiently large,

$$
X \approx N\big(np,\ np(1-p)\big)
$$

— a normal distribution with exactly the binomial's mean and variance. (Sums and averages tell the same story: the average $X/n$ is then approximately $N(p, \tfrac{p(1-p)}{n})$ — the sampling distribution of the sample proportion from earlier in this module.)

## 3. Worked Examples

### Example 1: the table, verified for $n = 4$

Binomial(4, 0.5) has PMF $\frac{1, 4, 6, 4, 1}{16}$. Mean: $\sum k \, p_k = \frac{0 + 4 + 12 + 12 + 4}{16} = 2 = np$. Variance: $E[X^2] - 4 = \frac{0 + 4 + 24 + 36 + 16}{16} - 4 = 5 - 4 = 1 = np(1-p)$. ✓

### Example 2: using the approximation

For $n = 100$ fair flips: $X \approx N(50, 25)$, so $\sigma = 5$. The 68–95–99.7 rule now applies to *coin counting*: about 95% of the time you'll see between 40 and 60 heads ($\mu \pm 2\sigma$) — a statement that would otherwise need summing 21 binomial coefficients.

### Example 3: a biased coin still turns Gaussian

$p = 0.2$ starts even more lopsided (a tall bar at 0). The bell still emerges — just needing larger $n$: the skewness of Binomial$(n, p)$ is $\frac{1-2p}{\sqrt{np(1-p)}}$, shrinking with $\sqrt n$. Rule of thumb: the approximation is decent once $np$ and $n(1-p)$ are both $\gtrsim 5$–10.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

def binomial_pmf(n, p):
    k = np.arange(n + 1)
    return k, np.array([comb(n, ki) * p**ki * (1 - p)**(n - ki) for ki in k])

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

### The video's table: means and variances

In [ ]:
p = 0.5
print(f"{'n':>4} {'mu = np':>8} {'var = np(1-p)':>14}   check vs the PMF")
for n in [1, 2, 3, 4, 10]:
    k, pmf = binomial_pmf(n, p)
    mean_pmf = np.sum(k * pmf)
    var_pmf = np.sum((k - mean_pmf)**2 * pmf)
    print(f"{n:>4} {n * p:>8} {n * p * (1 - p):>14}   ({mean_pmf:.2f}, {var_pmf:.2f})")

### From two bars to a bell: the PMF sequence

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, n in zip(axes, [1, 2, 3, 4, 10]):
    k, pmf = binomial_pmf(n, 0.5)
    ax.bar(k, pmf, color="tab:blue", edgecolor="black", alpha=0.85)
    ax.set_title(f"n = {n}\n$\\mu$={n*0.5}, $\\sigma^2$={n*0.25}")
    ax.set_xlabel("number of heads")
    ax.set_xticks(k if n <= 4 else k[::2])
    ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("probability")
plt.suptitle("The number of heads in n coin flips: the bell emerges")
plt.tight_layout()
plt.show()

### The limiting Gaussian, overlaid

For growing $n$, the binomial bars against the curve $N(np, np(1-p))$ — by $n = 100$ they are indistinguishable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, n in zip(axes, [10, 30, 100]):
    k, pmf = binomial_pmf(n, 0.5)
    mu, sigma = n * 0.5, np.sqrt(n * 0.25)
    x = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
    ax.bar(k, pmf, color="tab:blue", alpha=0.6, label="Binomial PMF")
    ax.plot(x, normal_pdf(x, mu, sigma), color="tab:red", linewidth=2,
            label=f"N({mu:.0f}, {sigma**2:.0f})")
    ax.set_title(f"n = {n}")
    ax.set_xlim(mu - 4 * sigma, mu + 4 * sigma)
    ax.set_xlabel("number of heads")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
axes[0].set_ylabel("probability")
plt.suptitle("As n grows, Binomial(n, 1/2) matches N(np, np(1-p))")
plt.tight_layout()
plt.show()

### Simulation view: flip coins, count heads, histogram

The theorem is about real coin experiments too: 100,000 repetitions of "flip 100 coins" produce a histogram sitting on the Gaussian curve.

In [ ]:
rng = np.random.default_rng(42)

n = 100
heads = rng.integers(0, 2, size=(100_000, n)).sum(axis=1)
mu, sigma = n * 0.5, np.sqrt(n * 0.25)

plt.figure(figsize=(9, 4.5))
plt.hist(heads, bins=np.arange(25, 76) - 0.5, density=True, color="tab:blue",
         alpha=0.6, label="100,000 experiments")
x = np.linspace(25, 75, 400)
plt.plot(x, normal_pdf(x, mu, sigma), color="tab:red", linewidth=2,
         label=f"N({mu:.0f}, {sigma**2:.0f})")
plt.title("Flip 100 coins, count heads, repeat: the CLT in the wild")
plt.xlabel("number of heads")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"simulated mean {heads.mean():.2f} (np = {mu}), "
      f"variance {heads.var():.2f} (np(1-p) = {sigma**2})")

## 5. Visualizations

### A biased coin: lopsided start, Gaussian finish

$p = 0.2$: at small $n$ the PMF leans hard left; by $n = 100$ the bell has won. Watch the skew drain away.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, n in zip(axes, [1, 5, 20, 100]):
    k, pmf = binomial_pmf(n, 0.2)
    mu, sigma = n * 0.2, np.sqrt(n * 0.16)
    ax.bar(k, pmf, color="tab:orange", alpha=0.7, label="PMF")
    if n >= 5:
        x = np.linspace(max(0, mu - 4 * sigma), mu + 4 * sigma, 300)
        ax.plot(x, normal_pdf(x, mu, sigma), color="tab:red", linewidth=2)
    ax.set_title(f"n = {n}  (skew {((1 - 2*0.2) / sigma):.2f})")
    ax.set_xlim(-0.8, min(n, mu + 4.5 * sigma))
    ax.set_xlabel("heads")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("probability")
plt.suptitle("Binomial(n, 0.2): even a biased coin's counts become Gaussian")
plt.tight_layout()
plt.show()

### How good is the approximation? The error vs n

Maximum gap between the binomial CDF and the normal CDF, for the fair and the biased coin — shrinking like $1/\sqrt{n}$, slower when $p$ is far from $\tfrac12$.

In [ ]:
def max_cdf_gap(n, p):
    k, pmf = binomial_pmf(n, p)
    binom_cdf = np.cumsum(pmf)
    mu, sigma = n * p, np.sqrt(n * p * (1 - p))
    # normal CDF at k + 0.5 via numerical integration of the pdf
    grid = np.linspace(mu - 8 * sigma, mu + 8 * sigma, 4001)
    pdf = normal_pdf(grid, mu, sigma)
    cdf_grid = np.cumsum(pdf) * (grid[1] - grid[0])
    normal_cdf = np.interp(k + 0.5, grid, cdf_grid)
    return np.max(np.abs(binom_cdf - normal_cdf))

ns = [5, 10, 20, 50, 100, 200, 500]
plt.figure(figsize=(8.5, 4.5))
for p, color in [(0.5, "tab:blue"), (0.2, "tab:orange")]:
    gaps = [max_cdf_gap(n, p) for n in ns]
    plt.plot(ns, gaps, "o-", color=color, label=f"p = {p}")
plt.xscale("log")
plt.yscale("log")
plt.title("Worst-case CDF error of the normal approximation")
plt.xlabel("n (log)")
plt.ylabel("max |Binomial CDF - Normal CDF| (log)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For $n = 10$ fair flips, compute $\mu$ and $\sigma^2$ from the formulas and verify against the PMF. Then use the 68–95–99.7 rule on the approximating normal: between which head-counts do ~95% of experiments land?

**Exercise 2.** For a biased coin with $p = 0.3$ and $n = 50$: what normal distribution approximates the head count? Check the rule of thumb ($np$, $n(1-p) \gtrsim 5$).

### Intermediate

**Exercise 3.** Compare $P(X = 50)$ exactly (binomial coefficient) with the normal density at 50 for $n = 100$, $p = 0.5$. Then compare $P(45 \le X \le 55)$ exactly with the normal probability of $[44.5, 55.5]$ (the *continuity correction*). How much does the half-unit shift help?

**Exercise 4.** It's about sums, not coins: repeat the morphing-PMF experiment with a fair **die** — plot the distribution of the sum of $n$ dice for $n = 1, 2, 5, 30$ (by convolution or simulation) with the matching normal overlay $N(3.5n, \tfrac{35}{12}n)$.

### Advanced

**Exercise 5.** How non-normal can we start? Take the ±\$100 game's two-spike distribution (as far from a bell as it gets) and plot the distribution of the *average* of $n$ plays for $n = 2, 10, 100$. Confirm the CLT still wins, and note what $\sigma/\sqrt{n}$ does to the axis scale.

**Exercise 6.** The classic application — polling error revisited: with $n = 1000$ and true $p = 0.52$, use the normal approximation $\hat{p} \approx N(p, \tfrac{p(1-p)}{n})$ to compute $P(\hat{p} < 0.5)$ analytically (via the standard normal CDF), and compare with the simulation answer from the sample-proportion notebook (~0.10).

## 7. Solutions

In [ ]:
# Exercise 1: n = 10 fair flips
k, pmf = binomial_pmf(10, 0.5)
mean_ = np.sum(k * pmf)
var_ = np.sum((k - mean_)**2 * pmf)
sigma_ = np.sqrt(var_)
print(f"Exercise 1: mu = {mean_} (np = 5), sigma^2 = {var_} (np(1-p) = 2.5)")
print(f"95% band: mu +- 2 sigma = [{mean_ - 2*sigma_:.1f}, {mean_ + 2*sigma_:.1f}]"
      f" -> roughly 2 to 8 heads")
exact = pmf[(k >= 2) & (k <= 8)].sum()
print(f"exact P(2 <= X <= 8) = {exact:.4f}")

In [ ]:
# Exercise 2: biased coin check
n, p = 50, 0.3
print(f"Exercise 2: X ~~ N({n*p}, {n*p*(1-p):.1f}), sigma = {np.sqrt(n*p*(1-p)):.2f}")
print(f"rule of thumb: np = {n*p}, n(1-p) = {n*(1-p)} -- both >> 5, approximation OK")

In [ ]:
# Exercise 3: continuity correction
n, p = 100, 0.5
mu, sigma = n * p, np.sqrt(n * p * (1 - p))

exact_50 = comb(100, 50) * 0.5**100
print(f"Exercise 3: P(X = 50) exact = {exact_50:.5f}, "
      f"normal density at 50 = {normal_pdf(50, mu, sigma):.5f}")

k, pmf = binomial_pmf(100, 0.5)
exact_45_55 = pmf[(k >= 45) & (k <= 55)].sum()

grid = np.linspace(mu - 8 * sigma, mu + 8 * sigma, 20001)
pdf = normal_pdf(grid, mu, sigma)
def normal_prob(a, b):
    mask = (grid >= a) & (grid <= b)
    return np.trapezoid(pdf[mask], grid[mask])

print(f"P(45 <= X <= 55) exact          = {exact_45_55:.4f}")
print(f"normal on [45, 55]  (naive)     = {normal_prob(45, 55):.4f}")
print(f"normal on [44.5, 55.5] (corrected) = {normal_prob(44.5, 55.5):.4f}")
print("The half-unit widening accounts for the bars' width -- and nearly nails it.")

In [ ]:
# Exercise 4: dice sums morph too
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, n in zip(axes, [1, 2, 5, 30]):
    sums = rng.integers(1, 7, size=(200_000, n)).sum(axis=1)
    mu_n, var_n = 3.5 * n, 35 / 12 * n
    lo, hi = sums.min(), sums.max()
    ax.hist(sums, bins=np.arange(lo, hi + 2) - 0.5, density=True,
            color="tab:green", alpha=0.7)
    x = np.linspace(mu_n - 4 * np.sqrt(var_n), mu_n + 4 * np.sqrt(var_n), 300)
    ax.plot(x, normal_pdf(x, mu_n, np.sqrt(var_n)), color="tab:red", linewidth=2)
    ax.set_title(f"sum of {n} dice")
    ax.grid(True, alpha=0.3)
plt.suptitle("Exercise 4: dice sums head for N(3.5n, 35n/12) -- flat start, bell finish")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 5: averages of the two-spike game
rng = np.random.default_rng(1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, n in zip(axes, [2, 10, 100]):
    avgs = rng.choice([-100.0, 100.0], size=(200_000, n)).mean(axis=1)
    ax.hist(avgs, bins=60, density=True, color="tab:purple", alpha=0.7)
    sigma_n = 100 / np.sqrt(n)
    x = np.linspace(-4 * sigma_n, 4 * sigma_n, 300)
    ax.plot(x, normal_pdf(x, 0, sigma_n), color="tab:red", linewidth=2)
    ax.set_title(f"average of {n} plays\n(std {sigma_n:.0f})")
    ax.grid(True, alpha=0.3)
plt.suptitle("Exercise 5: two spikes -> bell; the axis narrows like 100/sqrt(n)")
plt.tight_layout()
plt.show()

print("Even the least bell-like start becomes Gaussian, while sigma/sqrt(n)")
print("squeezes the spread: CLT gives the shape, LLN the destination.")

In [ ]:
# Exercise 6: polling error, analytically
n, p = 1000, 0.52
sigma_phat = np.sqrt(p * (1 - p) / n)
z = (0.5 - p) / sigma_phat

grid = np.linspace(-8, 8, 40001)
pdf = normal_pdf(grid, 0, 1)
cdf_at_z = np.trapezoid(pdf[grid <= z], grid[grid <= z])
print(f"Exercise 6: sigma of p_hat = {sigma_phat:.4f}, z = (0.5-0.52)/sigma = {z:.3f}")
print(f"P(p_hat < 0.5) ~ Phi({z:.3f}) = {cdf_at_z:.3f}")
print("The sample-proportion notebook's simulation gave ~0.10 -- the CLT turns")
print("that simulation into a one-line normal-table lookup.")

## 8. Conclusion

- The **central limit theorem**: sums (and averages) of many i.i.d. variables become normally distributed, *whatever* the starting shape — two coin-spikes included.
- The discrete showcase: the head count in $n$ flips is a sum of $n$ Bernoullis; its Binomial PMF morphs from two bars ($n=1$) into a recognizable bell by $n = 10$, with parameters $\mu = np$ and $\sigma^2 = np(1-p)$ (the video's table: $(0.5, 0.25), (1, 0.5), \dots, (5, 2.5)$).
- For large $n$: $X \approx N(np, np(1-p))$ — putting the whole normal toolbox (68–95–99.7, z-scores, one CDF) at the service of counting problems; a continuity correction sharpens it.
- Biased coins and dice obey too — skewed starts just need larger $n$ (rule of thumb: $np, n(1-p) \gtrsim 5$).
- Together with the LLN: the law says where sample means go ($\mu$); the CLT says *how they fluctuate on the way* ($N(\mu, \sigma^2/n)$).

Next: the CLT for **continuous** starting distributions — same magic, no bars required.